# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build a **vector** environment from `EnvConfig`, sample random actions, and inspect the plain dict records returned each step.

See [docs/guide.md](../docs/guide.md) for the full step contract.

## Imports

`EnvConfig` accepts any Gymnasium environment id through `group_id`. `make_vector_env` wraps Gymnasium and formats steps as `(result, metrics)`.

In [9]:
from tensordict import TensorDict as TD
from mouse_envs import EnvConfig, make_vector_env

## Configure the environment

- `num_envs=1` runs one CartPole instance.
- `max_episode_steps` caps episode length (truncation).
- `seed` fixes RNG for reproducibility.

In [10]:
cfg = EnvConfig(
    group_id="CartPole-v1",
    seed=0,
    num_envs=1,
    max_episode_steps=500,
)
env = make_vector_env(cfg)

## Rollout loop

Each `env.step` returns:

- **`result[i]`** — `time`, `observation` (dict: `discrete` / `continuous` / `image`), `reward`, `done`, `group_id`, `episode_index`, `reward_episodic`, optional `q_star` / `ns_params`
- **`metrics[i]`** — `episode_cum_reward` and `episode_length` lists for finishes on this step (empty if none)

Each **`actions[i]["action"]`** passed to `step()` is also a dict — `discrete` or `continuous`, matching the env's action space.

The **first** `step` call performs an internal reset (actions are ignored); see the guide for the initial frame (`time == 0`).

In [11]:
for step in range(1000):
    actions = env.sample_random_actions()
    result, metrics = env.step(actions)

    for i, r in enumerate(result):
        obs = {k: v.tolist() for k, v in TD(r["observation"]).items()}
        act = {k: v.tolist() for k, v in TD(actions[i]["action"]).items()}
        print(
            f"step={step:4d}  "
            f"group_id={r['group_id']}  "
            f"time={r['time'].item()}  "
            f"act={act}  "
            f"done={r['done'].item()}  "
            f"reward={r['reward'].item():.3f}  "
            f"reward_episodic={r['reward_episodic']:.3f}  "
            f"obs={obs}"
        )

step=   0  group_id=CartPole-v1  time=0  act={'discrete': [0]}  done=0  reward=0.000  reward_episodic=0.000  obs={'continuous': [-0.014679748564958572, 0.01799100451171398, 0.0375664159655571, -0.02614838071167469]}
step=   1  group_id=CartPole-v1  time=1  act={'discrete': [0]}  done=0  reward=1.000  reward_episodic=0.002  obs={'continuous': [-0.014319928362965584, -0.17764897644519806, 0.03704344481229782, 0.2781464755535126]}
step=   2  group_id=CartPole-v1  time=2  act={'discrete': [1]}  done=0  reward=1.000  reward_episodic=0.006  obs={'continuous': [-0.01787290722131729, 0.016925469040870667, 0.042606376111507416, -0.0026266854256391525]}
step=   3  group_id=CartPole-v1  time=3  act={'discrete': [1]}  done=0  reward=1.000  reward_episodic=0.010  obs={'continuous': [-0.017534399405121803, 0.21141129732131958, 0.042553842067718506, -0.2815681993961334]}
step=   4  group_id=CartPole-v1  time=4  act={'discrete': [1]}  done=0  reward=1.000  reward_episodic=0.014  obs={'continuous': [-0

## Cleanup

In [12]:
env.close()